In [46]:
from pathlib import Path

import pandas as pd
import polars as pl
import polars_h3 as plh3
import numpy as np
import more_itertools
from datetime import date
import yaml

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [2]:
RES = 8

In [3]:
path = f"{config['dirs']['output']}/{config['city']}"

In [4]:
pl.read_parquet(f"{path}/stops/00.parquet")

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8
1,4,2024-10-05 23:53:54 CEST,2024-10-06 00:04:27 CEST,10m 33s,47.476742,19.075131,0.0,0.0,2,8
1,5,2024-10-09 04:46:54 CEST,2024-10-09 04:58:36 CEST,11m 42s,47.476778,19.075128,3.0,4.0,2,8
…,…,…,…,…,…,…,…,…,…,…
6584,334,2024-11-17 11:07:57 CET,2024-11-17 11:24:17 CET,16m 20s,47.606873,19.270928,8.0,1.0,3,4
6584,335,2024-11-17 18:15:42 CET,2024-11-17 18:37:33 CET,21m 51s,47.60659,19.270803,14.0,25.0,5,4
6584,336,2024-11-17 21:11:36 CET,2024-11-17 21:26:20 CET,14m 44s,47.606829,19.270937,11.0,6.0,3,4


In [20]:
def load_data(i: int, res: int, path: str) -> pd.DataFrame:
    data = pl.read_parquet(f"{path}/{i:02x}.parquet")
    data = data.with_columns(
        plh3.latlng_to_cell(
            pl.col("latitude_mean"), pl.col("longitude_mean"), res
        ).alias(f"h3_{res}"),
        pl.col("interval_start").dt.to_string("%Y-%m-%d").alias("day"),
    )

    data = data.sort("device_id", "interval_start", descending=[False, False])
    return data.to_pandas()


def get_daily_stop_sequence(seq, res: int = 7) -> dict | float:
    result = {}
    for i, j in more_itertools.pairwise(seq[f"h3_{res}"]):
        if i == j or pd.isna(i) or pd.isna(j):
            continue
        if i not in result:
            result[i] = {}
        if j not in result[i]:
            result[i][j] = 0
        result[i][j] += 1

    if len(result) == 0:
        return np.nan
    else:
        return result


def create_df(x) -> pd.DataFrame:
    r = []
    for i in x:
        for k, v in i["connections"].items():
            for kk, vv in v.items():
                r.append((i["device_id"], i["day"], k, kk, vv))
    return pd.DataFrame.from_records(
        r, columns=["device_id", "day", "source", "target", "weight"]
    )

In [58]:
dummy_seq = pd.DataFrame(
    {
        "device_id": [1] * 7,
        "day": [date(2026, 5, 19)] * 5 + [date(2026, 5, 20)] * 2,
        "h3_7": [1, 2, 3, 2, 3, 1, 2],
        "latitude_mean": [47.42, 47.64, 47.23, 47.87, 47.25, 47.42, 47.64],
        "longitude_mean": [18.91, 19.12, 19.09, 19.31, 19.08, 18.91, 19.12],
    }
)

In [59]:
get_daily_stop_sequence(dummy_seq)

{1: {2: 2}, 2: {3: 2}, 3: {2: 1, 1: 1}}

In [24]:
def get_daily_stop_sequence_pl(seq, res: int) -> dict:
    return {}


get_daily_stop_sequence_pl(dummy_seq, 7)

{}

In [ ]:
def get_daily_stop_sequence_pl(seq: pl.DataFrame, hexagon_column: str) -> dict:
    transitions = (
        seq.select(
            [
                pl.col(hexagon_column).alias("src"),
                pl.col(hexagon_column).shift(-1).alias("dst"),
            ]
        )
        .filter(
            (pl.col("src") != pl.col("dst"))
            & pl.col("src").is_not_null()
            & pl.col("dst").is_not_null()
        )
        .group_by(["src", "dst"])
        .len()
    )

    result = {}

    for row in transitions.iter_rows(named=True):
        src, dst, count = row["src"], row["dst"], row["len"]

        result.setdefault(src, {})[dst] = count

    return result


get_daily_stop_sequence_pl(pl.from_pandas(dummy_seq), "h3_7")

{2: {3: 2}, 1: {2: 1}, 3: {2: 1}}

In [78]:
result = (
    (
        pl.from_pandas(dummy_seq)
        .with_columns(
            [pl.col("h3_7").shift(-1).over(["device_id", "day"]).alias("target")]
        )
        .filter((pl.col("h3_7") != pl.col("target")) & pl.col("target").is_not_null())
        .group_by(["device_id", "day", "h3_7", "target"])
        .len()
    )
    .rename({"h3_7": "source", "len": "weight"})
    .sort(["device_id", "day"])
)

In [85]:
result

device_id,day,source,target,weight
i64,date,i64,i64,u32
1,2026-05-19,3,2,1
1,2026-05-19,1,2,1
1,2026-05-19,2,3,2
1,2026-05-20,1,2,1


In [ ]:
# runs 14m for resolution 7
# runs 11m for resolution 8
# runs 13m for resolution 9
# runs 13m for resolution 10

Path(f"{path}/hexagon_network/{RES}").mkdir(parents=True, exist_ok=True)
for i in range(256):
    padf = load_data(i, RES, path + "/stops")
    dss = (
        padf.groupby(["device_id", "day"])
        .apply(get_daily_stop_sequence, include_groups=False, res=RES)
        .reset_index()
    )
    dss.rename({0: "connections"}, axis=1, inplace=True)
    dss = dss.query("connections.notna()").reset_index(drop=True)
    dss = create_df(dss.to_dict(orient="records"))
    dss.to_csv(f"{path}/hexagon_network/{RES}/connections_{i:02x}.csv", index=False)
    break

In [ ]:
connections = pl.DataFrame()
for i in range(256):
    temp = pl.read_csv(f"{path}/{RES}/connections_{i:02x}.csv")
    connections = pl.concat([connections, temp])

In [ ]:
network = (
    connections.group_by(["source", "target"])
    .agg(pl.col("weight").sum())
    .sort("source", "target")
)
network.write_csv(f"../output/network_hex{RES}/edges.csv")

In [ ]:
network_by_unique_device = (
    connections.group_by(["source", "target"])
    .agg(pl.col("device_id").unique().len().alias("weight"))
    .sort("source", "target")
)
network_by_unique_device.write_csv(
    f"../output/network_hex{RES}/edges_unique_device.csv"
)